### Dependencies

In [1]:
import pandas as pd 
import numpy as np
from utils import DesignValuesSheetUpdate, soil_limitstate_value

### Data overview

In [2]:
# prime_dt = pd.read_csv("./LayerSC/Layer10.csv")
# print(prime_dt,'\n')

# mean = prime_dt.mean()  
# sum = prime_dt.sum()   
# std_dev = prime_dt.loc[:, prime_dt.columns != 'W_content'].std(ddof=1)
# variance = prime_dt.loc[:, prime_dt.columns != 'W_content'].var(ddof=1)

# # print(prime_dt.columns)
# print('Standard Deviation:--------------\n',np.round(std_dev, 4), '\n')
# print('Variance:--------------\n',np.round(variance, 4), '\n')
# print('Mean:--------------\n',np.round(mean, 2), '\n')
# print('Sum:--------------\n',sum, '\n')

In [3]:
n_v_table = pd.read_csv("./Coefficients/v_coef_sheet.csv")
print(n_v_table[n_v_table['n'] == 6]['v_max'].values[0])

2.07


## Calculate limit state design values

In [8]:
def soilProps_Stat (path):
    """
    This function calculates various statistical properties of soil samples from a given CSV file path.
    It reads the data, calculates variance, mean, and standard deviation, and filters the data based on a condition.
    It also prints the characteristic value and other statistical properties.

    Parameters:
        path (str): The file path to the CSV file containing soil sample data.

    Returns:
        tuple: A tuple containing the cleaned data, characteristic value, ultimate limit state design value (rho_U), and serviceability limit state design value
        (rho_S).
    """
    prime_dt = pd.read_csv(path)

    Soil_Prop_dt = prime_dt['Wet_U_weight']
    Soil_Prop_dt.index.name = None
    Soil_Prop_dt = Soil_Prop_dt.to_frame()

    Soil_Prop_count = Soil_Prop_dt['Wet_U_weight'].count()
    print('count:',Soil_Prop_count)
    
    Soil_Prop_var = Soil_Prop_dt['Wet_U_weight'].var()
    print('OK ✅' if Soil_Prop_var < 0.05 else 'Failed ❌') ## todo
    
    Soil_Prop_mean = Soil_Prop_dt['Wet_U_weight'].mean()
    print('mean:',Soil_Prop_mean)

    if Soil_Prop_count < 6:
        print('the number of samples is less than 6 ⬇️')
        characteristic_value = Soil_Prop_mean
        print(f'characteristic value: {characteristic_value:.2f}')
        print('---------------------------------------------')
        return None, characteristic_value, None, None

    # v' checking
    n_v_table = pd.read_csv("./Coefficients/v_coef_sheet.csv")
    v_max = n_v_table[n_v_table['n'] == Soil_Prop_count]['v_max'].values[0]    

    if Soil_Prop_count >= 6 and Soil_Prop_var < 25:
        Sample_pass_cond = Soil_Prop_dt['Wet_U_weight'].std()*v_max
    elif Soil_Prop_count >= 25:
        Sample_pass_cond = Soil_Prop_dt['Wet_U_weight'].std(ddof=1)*v_max  

    print('[v]=', Sample_pass_cond) 


    Soil_Prop_dt['Ad'] = (Soil_Prop_dt['Wet_U_weight'] - Soil_Prop_mean).abs()

    Soil_Prop_dt['Check'] = Soil_Prop_dt['Ad'] < Sample_pass_cond

    clean_data = Soil_Prop_dt.loc[Soil_Prop_dt['Check'] == True]
    clean_data.Name = 'Clean_data'
    characteristic_value = clean_data['Wet_U_weight'].mean()    



    print('characteristic value:', np.round(characteristic_value, 2))
    print('---------------------------------------------')

    rho_U, rho_S = soil_limitstate_value(Soil_Prop_count, Soil_Prop_var, characteristic_value, "./Coefficients/t_coef_sheet.csv")

    return clean_data, characteristic_value, rho_U, rho_S 



#=============================================================================================================

a = soilProps_Stat("./LayerSC/Layer7.csv")

count: 5
OK ✅
mean: 1.908
the number of samples is less than 6 ⬇️
characteristic value: 1.91
---------------------------------------------


### Optional for Google Sheets
this code block is used for converting dataframe to a spreadsheet for a Google sheet file.

In [5]:
from utils import read_csv_files    

folder_path = "./LayerSC"  # Replace with the actual path to your folder
dataframes = read_csv_files(folder_path)

if dataframes:
    for filename, df in dataframes.items():
        print(f"🌟🌟 Calculation for {filename}")
        clean_data, characteristic_value, rho_U, rho_S = soilProps_Stat(f'{folder_path}/{filename}') 
        DesignValuesSheetUpdate('pysheetAuth.json', clean_data, characteristic_value, rho_U, rho_S, workSheetName=filename.replace('.csv', ''))
        print(f'{'=' * 150} \n \n')

🌟🌟 Calculation for Layer1.csv
count: 14
OK ✅
mean: 1.5021428571428574
[v]= 0.14356382353702993
characteristic value: 1.5
---------------------------------------------
Design values:
Ultimate limit state design value t (TTGH I):  1.77
Serviceability limit state design value t (TTGH II):  1.08
rho_ultimate: 0.4900640606404119
rho_serviceability: 0.29869249294270783
---------------------------------------------
gamma_I = 1.50(1 ± 0.4901)
gamma_II = 1.50(1 ± 0.2987)
Data updated in the worksheet: Layer1
 

🌟🌟 Calculation for Layer10.csv
count: 40
OK ✅
mean: 1.97025
[v]= 0.07373491810150666
characteristic value: 1.97
---------------------------------------------
Design values:
Ultimate limit state design value t (TTGH I):  1.682
Serviceability limit state design value t (TTGH II):  1.05
rho_ultimate: 0.26924318333758657
rho_serviceability: 0.16804219013277152
---------------------------------------------
gamma_I = 1.97(1 ± 0.2692)
gamma_II = 1.97(1 ± 0.1680)
Data updated in the worksheet: L